# 3.17 — Linear & Quadratic Discriminant Analysis

Linear and Quadratic Discriminant Analysis classify by fitting one Gaussian cloud per class, then comparing class log scores. LDA shares one covariance matrix across classes, which makes the boundary linear; QDA lets each class keep its own covariance, which makes the boundary curve when class shapes differ.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build discriminant analysis one idea at a time. Run each cell in order and read the printed intermediate values — every probability score is kept in log space and every matrix calculation is visible. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, means, covariances, inverses, and log scores.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any jittered toy data.

### 1. A class is summarized by a mean, covariance, and prior

Discriminant analysis begins with a modeling promise: inside each class, feature vectors look like draws from a multivariate Gaussian. That means each class needs a center `mu`, a shape matrix `Sigma`, and a prior probability `pi`. The center tells us where typical examples live; the covariance tells us which directions vary a lot; the prior tells us what we believed before seeing the new point.

In [ ]:
X0_w = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1]])  # class 0 cloud.
X1_w = np.array([[3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # class 1 cloud.
X_w = np.vstack([X0_w, X1_w])  # stack both classes for plotting.
y_w = np.array([0] * len(X0_w) + [1] * len(X1_w))  # matching class labels.
print("class counts:", np.bincount(y_w))  # priors come from these counts.
print("first class-0 point:", X0_w[0])  # inspect one observation.

▶ What you'll see: two balanced classes with four examples each, so both empirical priors are 0.5.

In [ ]:
mu0_w = X0_w.mean(axis=0)  # empirical class-0 mean.
mu1_w = X1_w.mean(axis=0)  # empirical class-1 mean.
S0_w = np.cov(X0_w, rowvar=False, bias=False)  # unbiased class-0 covariance.
S1_w = np.cov(X1_w, rowvar=False, bias=False)  # unbiased class-1 covariance.
pi0_w = len(X0_w) / len(X_w)  # empirical prior for class 0.
pi1_w = len(X1_w) / len(X_w)  # empirical prior for class 1.
print("mu0:", np.round(mu0_w, 3), "mu1:", np.round(mu1_w, 3))  # centers.
print("pi0, pi1:", pi0_w, pi1_w)  # priors.
assert np.allclose(mu0_w, [1.1, 1.025]) and np.allclose(mu1_w, [3.125, 3.125])

▶ What you'll see: the class means are far apart, so a point near `[1.1, 1.0]` should favor class 0.

In [ ]:
plt.figure(figsize=(4.4, 3.6))  # compact scatterplot.
plt.scatter(X0_w[:, 0], X0_w[:, 1], color="steelblue", label="class 0")  # class 0 points.
plt.scatter(X1_w[:, 0], X1_w[:, 1], color="darkorange", label="class 1")  # class 1 points.
plt.scatter([mu0_w[0], mu1_w[0]], [mu0_w[1], mu1_w[1]], marker="x", s=120, color="black", label="means")
plt.title("1: class means summarize Gaussian centers"); plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.show()

▶ What you'll see: two point clouds and their black mean markers; the means are the Gaussian centers used for scoring.

*Why it's done this way:* a Gaussian is the maximum-entropy shape once we commit to a mean and covariance, so LDA/QDA turn classification into estimating those few stable summaries instead of memorizing every training point.

### 2. Log Gaussian scores avoid tiny probabilities

The multivariate Gaussian density contains an exponential term. Multiplying raw densities can underflow, and the normalizing constants are awkward, so discriminant analysis compares log scores instead. For one class, the important pieces are a distance term, a volume term, and a prior term:

$$\log p(x\mid k)+\log\pi_k=-\tfrac12(x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k)-\tfrac12\log|\Sigma_k|+\log\pi_k + C.$$

In [ ]:
x_w = np.array([2.2, 2.0])  # a new point between the two clouds.
def log_gaussian_w(x, mu, S, pi):  # full class log score up to the shared constant.
    diff = x - mu  # displacement from the class center.
    Sinv = np.linalg.inv(S)  # precision matrix weights directions by inverse variance.
    return float(-0.5 * diff @ Sinv @ diff - 0.5 * np.log(np.linalg.det(S)) + np.log(pi))
score0_w = log_gaussian_w(x_w, mu0_w, S0_w, pi0_w)  # class-0 score.
score1_w = log_gaussian_w(x_w, mu1_w, S1_w, pi1_w)  # class-1 score.
print("log scores:", round(score0_w, 3), round(score1_w, 3))  # larger wins.

▶ What you'll see: class 1 has the larger log score for this middle-right point.

In [ ]:
quad0_w = float((x_w - mu0_w) @ np.linalg.inv(S0_w) @ (x_w - mu0_w))  # Mahalanobis distance squared to class 0.
quad1_w = float((x_w - mu1_w) @ np.linalg.inv(S1_w) @ (x_w - mu1_w))  # Mahalanobis distance squared to class 1.
print("Mahalanobis^2:", round(quad0_w, 3), round(quad1_w, 3))  # lower distance helps the score.
assert round(quad0_w, 3) == 295.528 and round(quad1_w, 3) == 37.750

▶ What you'll see: inverse covariance turns "distance" into a shape-aware distance, not plain Euclidean distance.

In [ ]:
plt.figure(figsize=(4.4, 3.2))  # show score comparison.
plt.bar(["class 0", "class 1"], [score0_w, score1_w], color=["steelblue", "darkorange"])  # larger log score wins.
plt.title("2: compare log Gaussian scores"); plt.ylabel("log score (larger is better)"); plt.show()

▶ What you'll see: the class-1 bar is higher because the point is more plausible under class 1's fitted Gaussian.

*Why it's done this way:* taking logs turns products into sums, keeps tiny densities numerically safe, and preserves the winner because log is monotone; the inverse covariance makes deviations in high-variance directions less surprising than deviations in tight directions.

### 3. LDA shares covariance, so the boundary is linear

LDA assumes all classes have the same covariance. When the same `Sigma` appears in every class score, the `x.T @ inv(Sigma) @ x` quadratic term cancels between classes. What remains is exactly the lesson's linear discriminant form:

$$\delta_k(x)=x^\top\Sigma^{-1}\mu_k-\tfrac12\mu_k^\top\Sigma^{-1}\mu_k+\log\pi_k.$$

In [ ]:
n0_w, n1_w = len(X0_w), len(X1_w)  # class sizes.
Sp_w = ((n0_w - 1) * S0_w + (n1_w - 1) * S1_w) / (n0_w + n1_w - 2)  # pooled covariance.
print("pooled covariance:\n", np.round(Sp_w, 3))  # one shared shape for both classes.
assert np.allclose(np.round(Sp_w, 3), [[0.058, -0.024], [-0.024, 0.059]])

▶ What you'll see: LDA replaces two class shapes with one pooled shape estimated from both groups.

In [ ]:
def lda_delta_w(x, mu, Sigma, pi):  # LDA linear log score without shared constants.
    Sinv = np.linalg.inv(Sigma)  # shared precision.
    return float(x @ Sinv @ mu - 0.5 * mu @ Sinv @ mu + np.log(pi))
lda0_w = lda_delta_w(x_w, mu0_w, Sp_w, pi0_w)  # LDA score for class 0.
lda1_w = lda_delta_w(x_w, mu1_w, Sp_w, pi1_w)  # LDA score for class 1.
print("LDA deltas:", round(lda0_w, 3), round(lda1_w, 3), "prediction:", int(lda1_w > lda0_w))
assert round(lda1_w - lda0_w, 3) == 0.740

▶ What you'll see: class 1 wins by a large positive margin under the shared-covariance rule.

In [ ]:
xx_w, yy_w = np.meshgrid(np.linspace(0.4, 3.8, 90), np.linspace(0.4, 3.8, 90))  # plotting grid.
grid_w = np.c_[xx_w.ravel(), yy_w.ravel()]  # flatten grid points.
Z_w = np.array([lda_delta_w(g, mu1_w, Sp_w, pi1_w) - lda_delta_w(g, mu0_w, Sp_w, pi0_w) for g in grid_w]).reshape(xx_w.shape)
plt.figure(figsize=(4.4, 3.6)); plt.contourf(xx_w, yy_w, Z_w > 0, alpha=0.25, levels=[-0.5, 0.5, 1.5], colors=["steelblue", "darkorange"])
plt.contour(xx_w, yy_w, Z_w, levels=[0], colors="black")  # decision boundary.
plt.scatter(X0_w[:, 0], X0_w[:, 1], color="steelblue"); plt.scatter(X1_w[:, 0], X1_w[:, 1], color="darkorange")
plt.title("3: LDA boundary is a line"); plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

▶ What you'll see: the black boundary is straight because the quadratic terms cancel under a shared covariance.

*Why it's done this way:* pooling covariance lowers variance in small samples and forces a simple linear boundary; that constraint is a regularizing choice, not a computational accident.

### 4. QDA keeps class covariances, so the boundary can curve

QDA removes LDA's shared-shape assumption. Each class keeps its own covariance, so the quadratic term no longer cancels in a score difference. That gives QDA more flexibility: it can create curved boundaries when one class is elongated or more spread out than another.

In [ ]:
X0q_w = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2]])  # wide horizontal class.
X1q_w = np.array([[-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # tall vertical class.
mu0q_w, mu1q_w = X0q_w.mean(axis=0), X1q_w.mean(axis=0)  # class centers.
S0q_w = np.cov(X0q_w, rowvar=False); S1q_w = np.cov(X1q_w, rowvar=False)  # class-specific shapes.
print("S0 diag:", np.round(np.diag(S0q_w), 3), "S1 diag:", np.round(np.diag(S1q_w), 3))
assert np.round(np.diag(S0q_w), 3).tolist() == [0.696, 0.030]

▶ What you'll see: class 0 varies mostly in x1, while class 1 varies mostly in x2.

In [ ]:
def qda_delta_w(x, mu, Sigma, pi):  # QDA log score up to shared constants.
    diff = x - mu  # displacement from the class center.
    Sinv = np.linalg.inv(Sigma)  # class-specific precision.
    return float(-0.5 * diff @ Sinv @ diff - 0.5 * np.log(np.linalg.det(Sigma)) + np.log(pi))
probe_w = np.array([0.75, 1.1])  # a point where shape matters.
q0_w = qda_delta_w(probe_w, mu0q_w, S0q_w, 0.5)  # class 0 QDA score.
q1_w = qda_delta_w(probe_w, mu1q_w, S1q_w, 0.5)  # class 1 QDA score.
print("QDA scores:", round(q0_w, 3), round(q1_w, 3), "prediction:", int(q1_w > q0_w))

▶ What you'll see: the winner depends on both center and covariance shape, not just nearest mean.

In [ ]:
xxq_w, yyq_w = np.meshgrid(np.linspace(-1.5, 1.5, 120), np.linspace(-0.5, 3.0, 120))  # grid.
gridq_w = np.c_[xxq_w.ravel(), yyq_w.ravel()]  # flatten grid.
Zq_w = np.array([qda_delta_w(g, mu1q_w, S1q_w, 0.5) - qda_delta_w(g, mu0q_w, S0q_w, 0.5) for g in gridq_w]).reshape(xxq_w.shape)
plt.figure(figsize=(4.4, 3.6)); plt.contourf(xxq_w, yyq_w, Zq_w > 0, alpha=0.25, levels=[-0.5, 0.5, 1.5], colors=["steelblue", "darkorange"])
plt.contour(xxq_w, yyq_w, Zq_w, levels=[0], colors="black")
plt.scatter(X0q_w[:, 0], X0q_w[:, 1], color="steelblue"); plt.scatter(X1q_w[:, 0], X1q_w[:, 1], color="darkorange")
plt.title("4: QDA boundary can curve"); plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

▶ What you'll see: the black boundary bends because each class has its own covariance geometry.

*Why it's done this way:* QDA is the same Gaussian-score idea with fewer constraints; keeping separate covariances lowers bias when shapes truly differ, but it spends many more parameters and can overfit small samples.

### 5. Priors move the boundary before the features speak

The `log pi_k` term is not decoration. If one class is much more common, it starts with a score advantage. A point must provide enough feature evidence to overcome that prior advantage. This is useful when prevalence is real, but dangerous if training priors differ from deployment priors.

In [ ]:
edge_w = np.array([2.0, 1.9])  # a point near the LDA boundary.
score_equal_w = np.array([lda_delta_w(edge_w, mu0_w, Sp_w, 0.5), lda_delta_w(edge_w, mu1_w, Sp_w, 0.5)])  # equal priors.
score_skew_w = np.array([lda_delta_w(edge_w, mu0_w, Sp_w, 0.8), lda_delta_w(edge_w, mu1_w, Sp_w, 0.2)])  # class 0 more common.
print("equal-prior scores:", np.round(score_equal_w, 3), "winner:", int(np.argmax(score_equal_w)))
print("skew-prior scores:", np.round(score_skew_w, 3), "winner:", int(np.argmax(score_skew_w)))

▶ What you'll see: changing only priors shifts the margin because log priors add directly to discriminant scores.

In [ ]:
prior_bonus_w = np.log(0.8) - np.log(0.5)  # class-0 score boost from changing prior 0.5 -> 0.8.
print("class-0 log prior boost:", round(prior_bonus_w, 3))
assert round(prior_bonus_w, 3) == 0.470
plt.figure(figsize=(4.4, 3.2)); plt.bar(["equal prior", "class 0 prior=0.8"], [score_equal_w[0] - score_equal_w[1], score_skew_w[0] - score_skew_w[1]], color=["gray", "seagreen"])
plt.axhline(0, color="black", linewidth=1); plt.title("5: priors shift the LDA margin"); plt.ylabel("class0 score - class1 score"); plt.show()

▶ What you'll see: the margin moves upward by the log-prior bonus for class 0.

*Why it's done this way:* Bayes' rule multiplies likelihood by prior probability, so log space adds `log prior`; the boundary should move when common classes are genuinely more likely, but validation must check that the prior reflects the future population.

### 6. Model selection is score plus cost, not raw fit alone

The lesson's prose emphasizes the full decision score: raw empirical loss, method cost, validation gap, and a stabilizing knob. For LDA/QDA this is the practical choice between a simpler shared-covariance model and a more flexible class-specific-covariance model.

In [ ]:
losses_w = np.array([0.268, 0.148, 0.522])  # verified per-example losses for this lesson's toy instance.
R_S_w = round(float(losses_w.mean()), 3)  # empirical risk is the rounded average used in the lesson arithmetic.
cost_w = 0.100  # complexity or operational cost.
score_w = R_S_w + cost_w  # selection score includes the cost.
print("R_S:", round(R_S_w, 3), "score:", round(score_w, 3))
assert round(R_S_w, 3) == 0.313 and round(score_w, 3) == 0.413

▶ What you'll see: the raw training average is 0.313, but the score used for selection is 0.413 after cost.

In [ ]:
flex_w = 0.457  # tempting more-flexible alternative.
gap_w = flex_w - score_w  # absolute gap.
rel_gap_w = gap_w / flex_w  # relative gap on the alternative's scale.
stable_w = 0.80 * score_w  # stabilizing knob reduces the decision score by 20%.
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3), "stable:", round(stable_w, 3))
assert round(gap_w, 3) == 0.044 and round(rel_gap_w, 3) == 0.096 and round(stable_w, 3) == 0.330

▶ What you'll see: the flexible alternative loses here, while the stabilized score is lowest.

In [ ]:
labels_w = ["baseline", "flexible", "stabilized"]  # three model-selection candidates.
scores_w = np.array([score_w, flex_w, stable_w])  # comparable scores.
plt.figure(figsize=(4.5, 3.2)); plt.bar(labels_w, scores_w, color=["gray", "darkorange", "seagreen"])
plt.ylabel("selection score (lower is better)"); plt.title("6: choose by the full score"); plt.show()
print("winner:", labels_w[int(np.argmin(scores_w))])

▶ What you'll see: the stabilized bar is lowest, so it is the model carried forward in this toy comparison.

*Why it's done this way:* QDA's flexibility can improve raw fit while increasing variance; adding an explicit cost and checking gaps protects the learner from confusing an attractive training fragment with a durable future decision.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with inline `# ->` checks, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Estimate class summaries

LDA and QDA start by summarizing each labeled cloud with counts, priors, means, and covariance
shapes. Here the two classes have equal counts but different centers and spreads.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_X = np.array([[0.0, 1.0], [1.0, 1.0], [1.0, 2.0], [2.0, 2.0],
                 [4.0, 3.0], [5.0, 4.0], [5.0, 5.0], [6.0, 5.0]])  # -> 8 points
t1_y = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # -> [0, 0, 0, 0, 1, 1, 1, 1]
t1_classes = np.unique(t1_y)  # -> [0, 1]
t1_counts = np.array([np.sum(t1_y == t1_k) for t1_k in t1_classes])  # -> [4, 4]
t1_priors = t1_counts / t1_y.size  # -> [0.5, 0.5]
t1_means = np.vstack([t1_X[t1_y == t1_k].mean(axis=0) for t1_k in t1_classes])  # -> [[1.0, 1.5], [5.0, 4.25]]
t1_cov0 = np.cov(t1_X[t1_y == 0], rowvar=False, bias=False)  # -> [[0.667, 0.333], [0.333, 0.333]]
t1_cov1 = np.cov(t1_X[t1_y == 1], rowvar=False, bias=False)  # -> [[0.667, 0.667], [0.667, 0.917]]
print("classes:", t1_classes.tolist())  # -> [0, 1]
print("counts:", t1_counts.tolist())  # -> [4, 4]
print("priors:", t1_priors.tolist())  # -> [0.5, 0.5]
print("means:\n", np.round(t1_means, 3))  # -> [[1.0, 1.5], [5.0, 4.25]]
print("class-0 covariance:\n", np.round(t1_cov0, 3))  # -> [[0.667, 0.333], [0.333, 0.333]]
print("class-1 covariance:\n", np.round(t1_cov1, 3))  # -> [[0.667, 0.667], [0.667, 0.917]]
assert np.allclose(t1_priors, [0.5, 0.5])
assert np.allclose(np.round(t1_means, 2), [[1.0, 1.5], [5.0, 4.25]])

plt.figure(figsize=(4.8, 3.4))
plt.scatter(t1_X[t1_y == 0, 0], t1_X[t1_y == 0, 1], color="steelblue", label="class 0")
plt.scatter(t1_X[t1_y == 1, 0], t1_X[t1_y == 1, 1], color="darkorange", label="class 1")
plt.scatter(t1_means[:, 0], t1_means[:, 1], marker="x", s=140, color="black", label="means")
plt.title("Toy 1 · class summaries")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

▶ What you'll see: two balanced classes, black mean markers at `(1.0, 1.5)` and `(5.0, 4.25)`, and printed covariance shapes.

### ✍️ Toy 2 · Log Gaussian score pieces

A discriminant score is built from a shape-aware distance, a covariance-volume penalty, and a log
prior. Keeping those pieces visible makes the larger-score rule less mysterious.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_x = np.array([2.0, 2.5])  # -> [2.0, 2.5]
t2_mu = np.array([1.0, 2.0])  # -> [1.0, 2.0]
t2_cov = np.array([[2.0, 0.5], [0.5, 1.0]])  # -> [[2.0, 0.5], [0.5, 1.0]]
t2_prior = 0.6  # -> 0.6
t2_diff = t2_x - t2_mu  # -> [1.0, 0.5]
t2_inv = np.linalg.inv(t2_cov)  # -> [[0.571, -0.286], [-0.286, 1.143]]
t2_det = float(np.linalg.det(t2_cov))  # -> 1.75
t2_quad = float(t2_diff @ t2_inv @ t2_diff)  # -> 0.5714285714
t2_logdet = float(np.log(t2_det))  # -> 0.5596157879
t2_distance_piece = -0.5 * t2_quad  # -> -0.2857142857
t2_volume_piece = -0.5 * t2_logdet  # -> -0.2798078940
t2_prior_piece = float(np.log(t2_prior))  # -> -0.5108256238
t2_score = t2_distance_piece + t2_volume_piece + t2_prior_piece  # -> -1.0763478034
print("diff:", t2_diff.tolist())  # -> [1.0, 0.5]
print("inverse covariance:\n", np.round(t2_inv, 3))  # -> [[0.571, -0.286], [-0.286, 1.143]]
print("determinant:", round(t2_det, 3))  # -> 1.75
print("Mahalanobis^2:", round(t2_quad, 3))  # -> 0.571
print("score pieces:", np.round([t2_distance_piece, t2_volume_piece, t2_prior_piece], 3))  # -> [-0.286, -0.28, -0.511]
print("log score:", round(t2_score, 3))  # -> -1.076
assert round(t2_score, 3) == -1.076

plt.figure(figsize=(4.6, 3.0))
plt.bar(["distance", "volume", "prior"], [t2_distance_piece, t2_volume_piece, t2_prior_piece], color=["teal", "purple", "gray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 2 · log-score ingredients")
plt.ylabel("additive contribution")
plt.show()

▶ What you'll see: three negative score contributions that add to the printed log score `-1.076`.

### ✍️ Toy 3 · Pool covariance for an LDA line

LDA replaces class-specific shapes with one pooled covariance. Once that shared shape is inverted,
the class-1-vs-class-0 margin is a straight-line score.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_X0 = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])  # -> class 0 triangle
t3_X1 = np.array([[2.0, 2.0], [3.0, 2.0], [2.0, 3.0]])  # -> class 1 shifted triangle
t3_X = np.vstack([t3_X0, t3_X1])  # -> 6 total points
t3_y = np.array([0, 0, 0, 1, 1, 1])  # -> [0, 0, 0, 1, 1, 1]
t3_means = np.vstack([t3_X[t3_y == t3_k].mean(axis=0) for t3_k in [0, 1]])  # -> [[0.333, 0.333], [2.333, 2.333]]
t3_cov0 = np.cov(t3_X0, rowvar=False, bias=False)  # -> [[0.333, -0.167], [-0.167, 0.333]]
t3_cov1 = np.cov(t3_X1, rowvar=False, bias=False)  # -> [[0.333, -0.167], [-0.167, 0.333]]
t3_pooled = ((t3_X0.shape[0] - 1) * t3_cov0 + (t3_X1.shape[0] - 1) * t3_cov1) / (t3_X.shape[0] - 2)  # -> same shared covariance
t3_inv = np.linalg.inv(t3_pooled)  # -> [[4.0, 2.0], [2.0, 4.0]]
t3_priors = np.array([0.5, 0.5])  # -> [0.5, 0.5]
t3_query = np.array([1.5, 1.5])  # -> [1.5, 1.5]
t3_scores = np.array([t3_query @ t3_inv @ t3_means[t3_k] - 0.5 * t3_means[t3_k] @ t3_inv @ t3_means[t3_k] + np.log(t3_priors[t3_k]) for t3_k in [0, 1]])  # -> [4.64, 8.64]
t3_margin = float(t3_scores[1] - t3_scores[0])  # -> 4.0
t3_weight = t3_inv @ (t3_means[1] - t3_means[0])  # -> [12.0, 12.0]
t3_intercept = float(-0.5 * t3_means[1] @ t3_inv @ t3_means[1] + 0.5 * t3_means[0] @ t3_inv @ t3_means[0])  # -> -32.0
print("means:\n", np.round(t3_means, 3))  # -> [[0.333, 0.333], [2.333, 2.333]]
print("pooled covariance:\n", np.round(t3_pooled, 3))  # -> [[0.333, -0.167], [-0.167, 0.333]]
print("inverse pooled covariance:\n", np.round(t3_inv, 3))  # -> [[4.0, 2.0], [2.0, 4.0]]
print("LDA scores:", np.round(t3_scores, 3).tolist())  # -> [4.64, 8.64]
print("class-1 margin:", round(t3_margin, 3))  # -> 4.0
assert round(t3_margin, 3) == 4.0

plt.figure(figsize=(4.6, 3.4))
plt.scatter(t3_X0[:, 0], t3_X0[:, 1], color="steelblue", label="class 0")
plt.scatter(t3_X1[:, 0], t3_X1[:, 1], color="darkorange", label="class 1")
plt.scatter([t3_query[0]], [t3_query[1]], marker="*", s=150, color="black", label="query")
t3_line_x = np.linspace(-0.2, 3.2, 80)
t3_line_y = -(t3_weight[0] * t3_line_x + t3_intercept) / t3_weight[1]
plt.plot(t3_line_x, t3_line_y, color="black", linestyle="--", label="margin 0")
plt.title("Toy 3 · shared covariance gives a line")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

▶ What you'll see: the query sits on the class-1 side of a straight dashed LDA boundary.

### ✍️ Toy 4 · QDA keeps separate covariance

QDA scores each class with its own inverse covariance and determinant. The same point can be pulled
toward the class whose ellipse makes it more typical.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_X0 = np.array([[-2.0, 0.0], [-1.0, 0.5], [1.0, -0.5], [2.0, 0.0]])  # -> horizontal class
t4_X1 = np.array([[0.0, 1.0], [0.5, 2.0], [-0.5, 3.0], [0.0, 4.0]])  # -> vertical class
t4_X = np.vstack([t4_X0, t4_X1])  # -> 8 points
t4_y = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # -> balanced labels
t4_means = np.vstack([t4_X[t4_y == t4_k].mean(axis=0) for t4_k in [0, 1]])  # -> [[0.0, 0.0], [0.0, 2.5]]
t4_covs = np.array([np.cov(t4_X[t4_y == t4_k], rowvar=False, bias=False) for t4_k in [0, 1]])  # -> different shapes
t4_priors = np.array([0.5, 0.5])  # -> [0.5, 0.5]
t4_query = np.array([1.0, 1.2])  # -> [1.0, 1.2]
t4_scores = []
for t4_k in [0, 1]:
    t4_diff = t4_query - t4_means[t4_k]  # -> class-specific displacement
    t4_inv = np.linalg.inv(t4_covs[t4_k])  # -> class-specific precision
    t4_quad = float(t4_diff @ t4_inv @ t4_diff)  # -> Mahalanobis distance for class k
    t4_logdet = float(np.log(np.linalg.det(t4_covs[t4_k])))  # -> log determinant for class k
    t4_score = float(np.log(t4_priors[t4_k]) - 0.5 * (t4_quad + t4_logdet))  # -> QDA score without shared constant
    t4_scores.append(t4_score)
    print("class", t4_k, "diff", np.round(t4_diff, 3), "quad", round(t4_quad, 3), "score", round(t4_score, 3))
t4_scores = np.array(t4_scores)  # -> [-6.775, -3.03]
t4_pred = int(np.argmax(t4_scores))  # -> 1
print("means:\n", np.round(t4_means, 3))  # -> [[0.0, 0.0], [0.0, 2.5]]
print("covariance determinants:", np.round([np.linalg.det(t4_covs[0]), np.linalg.det(t4_covs[1])], 3).tolist())  # -> [0.444, 0.25]
print("QDA scores:", np.round(t4_scores, 3).tolist())  # -> [-6.775, -3.03]
print("prediction:", t4_pred)  # -> 1
assert t4_pred == 1

plt.figure(figsize=(4.7, 3.5))
plt.scatter(t4_X0[:, 0], t4_X0[:, 1], color="steelblue", label="class 0")
plt.scatter(t4_X1[:, 0], t4_X1[:, 1], color="darkorange", label="class 1")
plt.scatter([t4_query[0]], [t4_query[1]], marker="*", s=150, color="black", label="query")
plt.bar([2.8, 3.2], [t4_scores[0], t4_scores[1]], width=0.25, color=["steelblue", "darkorange"], alpha=0.7)
plt.title("Toy 4 · separate QDA scores")
plt.xlabel("x1 plus score bars at right")
plt.ylabel("x2 / log score")
plt.legend()
plt.show()

▶ What you'll see: class 1 wins because its separate covariance gives the query the higher QDA score.

### ✍️ Toy 5 · Priors shift the margin

The log prior adds directly to each class score. Here the feature evidence slightly favors class 1
under equal priors, but a strong class-0 prior flips the winner.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_log_likelihoods = np.array([-1.1, -0.9])  # -> class 1 is better by 0.2 before priors
t5_equal_priors = np.array([0.5, 0.5])  # -> equal prevalence
t5_skew_priors = np.array([0.8, 0.2])  # -> class 0 is much more common
t5_equal_scores = t5_log_likelihoods + np.log(t5_equal_priors)  # -> [-1.793, -1.593]
t5_skew_scores = t5_log_likelihoods + np.log(t5_skew_priors)  # -> [-1.323, -2.509]
t5_equal_margin = float(t5_equal_scores[0] - t5_equal_scores[1])  # -> -0.2
t5_skew_margin = float(t5_skew_scores[0] - t5_skew_scores[1])  # -> 1.1862943611
t5_equal_pred = int(np.argmax(t5_equal_scores))  # -> 1
t5_skew_pred = int(np.argmax(t5_skew_scores))  # -> 0
print("log likelihoods:", t5_log_likelihoods.tolist())  # -> [-1.1, -0.9]
print("equal-prior scores:", np.round(t5_equal_scores, 3).tolist())  # -> [-1.793, -1.593]
print("skew-prior scores:", np.round(t5_skew_scores, 3).tolist())  # -> [-1.323, -2.509]
print("equal margin class0-class1:", round(t5_equal_margin, 3))  # -> -0.2
print("skew margin class0-class1:", round(t5_skew_margin, 3))  # -> 1.186
print("predictions:", t5_equal_pred, t5_skew_pred)  # -> 1 0
assert t5_equal_pred == 1 and t5_skew_pred == 0

plt.figure(figsize=(4.6, 3.0))
plt.bar(["equal priors", "class0 prior 0.8"], [t5_equal_margin, t5_skew_margin], color=["gray", "seagreen"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 5 · priors move the margin")
plt.ylabel("class0 score - class1 score")
plt.show()

▶ What you'll see: the margin crosses from negative to positive when the class-0 prior becomes large.

### ✍️ Toy 6 · Compare the complete selection score

Model selection in the lesson adds cost, checks gaps, and may apply a stabilizing knob. The lowest
complete score, not the lowest raw loss alone, wins.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_losses = np.array([0.18, 0.22, 0.30, 0.20, 0.26, 0.24])  # -> six per-example losses
t6_risk = float(t6_losses.mean())  # -> 0.2333333333
t6_cost = 0.07  # -> 0.07
t6_score = t6_risk + t6_cost  # -> 0.3033333333
t6_flexible = 0.34  # -> 0.34
t6_gap = t6_flexible - t6_score  # -> 0.0366666667
t6_relative_gap = t6_gap / t6_flexible  # -> 0.1078431373
t6_stabilized = 0.85 * t6_score  # -> 0.2578333333
t6_labels = np.array(["baseline", "flexible", "stabilized"])  # -> three candidates
t6_scores = np.array([t6_score, t6_flexible, t6_stabilized])  # -> [0.303, 0.34, 0.258]
t6_winner = int(np.argmin(t6_scores))  # -> 2
print("losses:", t6_losses.tolist())  # -> [0.18, 0.22, 0.3, 0.2, 0.26, 0.24]
print("risk:", round(t6_risk, 3))  # -> 0.233
print("score with cost:", round(t6_score, 3))  # -> 0.303
print("gap:", round(t6_gap, 3), "relative:", round(t6_relative_gap, 3))  # -> 0.037 0.108
print("stabilized score:", round(t6_stabilized, 3))  # -> 0.258
print("winner:", t6_labels[t6_winner])  # -> stabilized
assert t6_labels[t6_winner] == "stabilized"

plt.figure(figsize=(4.8, 3.0))
plt.bar(t6_labels, t6_scores, color=["gray", "darkorange", "seagreen"])
plt.title("Toy 6 · full score chooses the model")
plt.ylabel("lower is better")
plt.show()

▶ What you'll see: adding cost and stabilization makes the green stabilized bar the winner.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for Gaussian parameters, matrix algebra, grids, and assertions.
import matplotlib.pyplot as plt  # load Matplotlib for scatterplots, contours, bars, and diagnostics.
np.random.seed(0)  # fix the global seed so every example is reproducible.

def fit_gaussian_params(X, y):  # estimate class means, covariances, and priors from labeled data.
    classes = np.unique(y)  # sorted class labels.
    mus, covs, priors = [], [], []  # storage for parameters.
    for c in classes:  # fit one Gaussian per class.
        Xc = X[y == c]  # examples in this class.
        mus.append(Xc.mean(axis=0))  # empirical center.
        covs.append(np.cov(Xc, rowvar=False, bias=False))  # unbiased covariance.
        priors.append(len(Xc) / len(X))  # empirical prior.
    return classes, np.array(mus), np.array(covs), np.array(priors)  # return arrays for scoring.

def pooled_covariance(X, y):  # compute the shared covariance used by LDA.
    classes = np.unique(y)  # class labels.
    p = X.shape[1]  # number of features.
    total = np.zeros((p, p))  # weighted covariance accumulator.
    denom = 0  # degrees-of-freedom accumulator.
    for c in classes:  # combine within-class covariance matrices.
        Xc = X[y == c]  # class examples.
        total += (len(Xc) - 1) * np.cov(Xc, rowvar=False, bias=False)  # within-class scatter.
        denom += len(Xc) - 1  # add degrees of freedom.
    return total / denom  # pooled covariance estimate.

def lda_scores(Xnew, mus, Sigma, priors):  # compute LDA discriminant scores for many points.
    Xnew = np.atleast_2d(Xnew)  # accept one point or many.
    Sinv = np.linalg.inv(Sigma)  # shared precision matrix.
    out = []  # collect one score column per class.
    for mu, pi in zip(mus, priors):  # score each class.
        out.append(Xnew @ Sinv @ mu - 0.5 * mu @ Sinv @ mu + np.log(pi))  # linear delta.
    return np.vstack(out).T  # rows are points, columns are classes.

def qda_scores(Xnew, mus, covs, priors, reg=0.0):  # compute QDA log scores for many points.
    Xnew = np.atleast_2d(Xnew)  # accept one point or many.
    out = []  # collect score columns.
    for mu, S, pi in zip(mus, covs, priors):  # score each class with its own covariance.
        Sreg = S + reg * np.eye(S.shape[0])  # optional diagonal stabilization.
        Sinv = np.linalg.inv(Sreg)  # class precision.
        diffs = Xnew - mu  # centered points.
        quad = np.sum((diffs @ Sinv) * diffs, axis=1)  # Mahalanobis squared distance.
        out.append(-0.5 * quad - 0.5 * np.log(np.linalg.det(Sreg)) + np.log(pi))  # QDA delta.
    return np.vstack(out).T  # rows are points, columns are classes.

def draw_boundary(X, y, score_fn, title):  # visualize a two-class decision boundary from a score function.
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 120)  # horizontal grid.
    x2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 120)  # vertical grid.
    xx, yy = np.meshgrid(x1, x2)  # mesh for contouring.
    grid = np.c_[xx.ravel(), yy.ravel()]  # flatten grid to points.
    margin = score_fn(grid)[:, 1] - score_fn(grid)[:, 0]  # class-1 minus class-0 score.
    plt.figure(figsize=(4.8, 3.7))  # compact boundary plot.
    plt.contourf(xx, yy, (margin.reshape(xx.shape) > 0), alpha=0.25, levels=[-0.5, 0.5, 1.5], colors=["steelblue", "darkorange"])  # regions.
    plt.contour(xx, yy, margin.reshape(xx.shape), levels=[0], colors="black")  # boundary.
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color="steelblue", label="class 0")  # class 0 points.
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color="darkorange", label="class 1")  # class 1 points.
    plt.title(title)  # title the plot.
    plt.xlabel("x1")  # x-axis label.
    plt.ylabel("x2")  # y-axis label.
    plt.legend()  # show labels.
    plt.show()  # display the boundary.

## 🟢 Basics (warm-up)

### Basic 1 — Build a tiny labeled dataset

**Goal.** Create two small feature clouds with labels, because LDA and QDA start from labeled examples and summarize each class separately. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # two visible groups.
y_b1 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # first four points are class 0, last four are class 1.
print("X shape:", X_b1.shape, "class counts:", np.bincount(y_b1))  # inspect size and balance.
assert X_b1.shape == (8, 2) and np.bincount(y_b1).tolist() == [4, 4]

▶ What you'll see: eight two-dimensional examples split evenly across two classes.

In [ ]:
plt.figure(figsize=(4, 3))  # create a compact scatterplot.
plt.scatter(X_b1[y_b1 == 0, 0], X_b1[y_b1 == 0, 1], color="steelblue", label="class 0")  # class 0.
plt.scatter(X_b1[y_b1 == 1, 0], X_b1[y_b1 == 1, 1], color="darkorange", label="class 1")  # class 1.
plt.title("Basic 1: labeled Gaussian-looking clouds"); plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.show()

▶ What you'll see: two separated clusters, making the Gaussian-class idea easy to inspect.

👀 Takeaway: discriminant analysis fits one probability model per class, so labels define which points estimate which Gaussian.

### Basic 2 — Compute class priors

**Goal.** Estimate prior probabilities from counts, because Bayes scoring should know how common each class is before seeing features. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[0.9, 1.0], [1.2, 1.1], [1.1, 0.8], [3.0, 3.1], [3.2, 2.9]])  # imbalanced toy data.
y_b2 = np.array([0, 0, 0, 1, 1])  # three examples from class 0 and two from class 1.
counts_b2 = np.bincount(y_b2)  # count examples per class.
print("counts:", counts_b2)  # inspect raw evidence for priors.

▶ What you'll see: class 0 appears three times and class 1 appears twice.

In [ ]:
priors_b2 = counts_b2 / counts_b2.sum()  # empirical priors.
print("priors:", priors_b2, "log priors:", np.round(np.log(priors_b2), 3))  # inspect additive log terms.
assert np.allclose(priors_b2, [0.6, 0.4])
plt.figure(figsize=(4, 3)); plt.bar(["class 0", "class 1"], priors_b2, color=["steelblue", "darkorange"])
plt.title("Basic 2: empirical priors"); plt.ylabel("prior probability"); plt.show()

▶ What you'll see: the class with more training examples starts with a higher prior score.

👀 Takeaway: priors are simple count proportions, but in log scores they become direct additive advantages.

### Basic 3 — Compute class means

**Goal.** Find each Gaussian center, because the mean is where that class's density is highest before covariance shape is considered. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # two classes.
y_b3 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
mu0_b3 = X_b3[y_b3 == 0].mean(axis=0)  # class-0 center.
mu1_b3 = X_b3[y_b3 == 1].mean(axis=0)  # class-1 center.
print("means:", np.round(mu0_b3, 3), np.round(mu1_b3, 3))  # inspect centers.
assert np.allclose(mu0_b3, [1.1, 1.025]) and np.allclose(mu1_b3, [3.125, 3.125])

▶ What you'll see: class 0 is centered near `(1.1, 1.025)` and class 1 near `(3.125, 3.125)`.

In [ ]:
plt.figure(figsize=(4, 3)); plt.scatter(X_b3[:, 0], X_b3[:, 1], c=y_b3, cmap="coolwarm")  # plot data.
plt.scatter([mu0_b3[0], mu1_b3[0]], [mu0_b3[1], mu1_b3[1]], marker="x", s=130, color="black")  # plot means.
plt.title("Basic 3: class centers"); plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

▶ What you'll see: black X markers at the centers of the two clouds.

👀 Takeaway: the class mean is the location parameter used by both LDA and QDA.

### Basic 4 — Compute one covariance matrix

**Goal.** Measure spread and correlation within one class, because covariance tells the Gaussian which directions are ordinary and which are surprising. We build it in 2 steps.

In [ ]:
X0_b4 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1]])  # class-0 examples.
S_b4 = np.cov(X0_b4, rowvar=False, bias=False)  # unbiased covariance across the two features.
print("covariance:\n", np.round(S_b4, 4))  # inspect variances and covariance.
assert np.allclose(np.round(S_b4, 4), [[0.0667, -0.0367], [-0.0367, 0.0292]])

▶ What you'll see: diagonal entries are feature variances, and the off-diagonal entry is negative correlation.

In [ ]:
plt.figure(figsize=(4, 3)); plt.scatter(X0_b4[:, 0], X0_b4[:, 1], color="steelblue")  # plot class points.
plt.axvline(X0_b4[:, 0].mean(), color="black", linestyle="--"); plt.axhline(X0_b4[:, 1].mean(), color="black", linestyle="--")
plt.title("Basic 4: spread around the mean"); plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

▶ What you'll see: covariance summarizes how points spread around the dashed mean lines.

👀 Takeaway: covariance is the Gaussian shape parameter; LDA pools it, while QDA keeps one per class.

### Basic 5 — Pool covariances for LDA

**Goal.** Combine class covariances into one shared covariance, because LDA assumes every class has the same shape. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # data.
y_b5 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
Sp_b5 = pooled_covariance(X_b5, y_b5)  # shared LDA covariance.
print("pooled covariance:\n", np.round(Sp_b5, 3))  # inspect pooled shape.
assert np.allclose(np.round(Sp_b5, 3), [[0.058, -0.024], [-0.024, 0.059]])

▶ What you'll see: one covariance matrix replaces separate within-class estimates.

In [ ]:
plt.figure(figsize=(4, 3)); plt.imshow(Sp_b5, cmap="viridis")  # visualize entries.
plt.colorbar(label="covariance value"); plt.xticks([0, 1], ["x1", "x2"]); plt.yticks([0, 1], ["x1", "x2"])
plt.title("Basic 5: pooled covariance for LDA"); plt.show()

▶ What you'll see: a small heatmap of the shared covariance used by every class score.

👀 Takeaway: LDA trades class-specific shape for a lower-variance shared estimate.

### Basic 6 — Score one point with LDA

**Goal.** Apply the linear discriminant formula to one point, because classification is just choosing the class with the largest score. We build it in 3 steps.

In [ ]:
X_b6 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # data.
y_b6 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
classes_b6, mus_b6, covs_b6, priors_b6 = fit_gaussian_params(X_b6, y_b6)  # estimate summaries.
Sp_b6 = pooled_covariance(X_b6, y_b6)  # shared covariance.
print("priors:", priors_b6)  # inspect prior terms.

▶ What you'll see: both classes have prior 0.5.

In [ ]:
x_b6 = np.array([2.2, 2.0])  # point to classify.
scores_b6 = lda_scores(x_b6, mus_b6, Sp_b6, priors_b6)[0]  # LDA scores.
print("LDA scores:", np.round(scores_b6, 3), "prediction:", int(np.argmax(scores_b6)))  # larger wins.
assert np.round(scores_b6[1] - scores_b6[0], 3) == 0.740

▶ What you'll see: class 1 has the larger discriminant score for this point.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["class 0", "class 1"], scores_b6, color=["steelblue", "darkorange"])
plt.title("Basic 6: LDA score comparison"); plt.ylabel("delta_k(x)"); plt.show()

▶ What you'll see: the taller bar identifies the predicted class.

👀 Takeaway: LDA prediction is an argmax over linear log scores.

### Basic 7 — Compute a Mahalanobis distance

**Goal.** Replace plain distance with covariance-aware distance, because a deviation is less surprising along a high-variance direction. We build it in 2 steps.

In [ ]:
x_b7 = np.array([2.2, 2.0])  # point being evaluated.
mu_b7 = np.array([1.1, 1.025])  # class center.
S_b7 = np.array([[0.058, -0.024], [-0.024, 0.059]])  # shared covariance approximation.
diff_b7 = x_b7 - mu_b7  # displacement from center.
print("diff:", np.round(diff_b7, 3))  # inspect raw displacement.

▶ What you'll see: the point is about one unit away from the class-0 mean in both coordinates.

In [ ]:
maha_b7 = float(diff_b7 @ np.linalg.inv(S_b7) @ diff_b7)  # squared Mahalanobis distance.
euclid_b7 = float(diff_b7 @ diff_b7)  # squared Euclidean distance.
print("Euclidean^2:", round(euclid_b7, 3), "Mahalanobis^2:", round(maha_b7, 3))  # compare metrics.
plt.figure(figsize=(4, 3)); plt.bar(["Euclidean²", "Mahalanobis²"], [euclid_b7, maha_b7], color=["gray", "purple"])
plt.title("Basic 7: covariance-aware distance"); plt.show()

▶ What you'll see: Mahalanobis distance is much larger because the class covariance is tight.

👀 Takeaway: Gaussian scores measure distance in units of class variance, not raw coordinate units.

### Basic 8 — Keep QDA class shapes separate

**Goal.** Estimate one covariance per class, because QDA allows different Gaussian shapes for different labels. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2], [-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # shape-different classes.
y_b8 = np.array([0] * 6 + [1] * 6)  # labels.
classes_b8, mus_b8, covs_b8, priors_b8 = fit_gaussian_params(X_b8, y_b8)  # class-specific parameters.
print("covariance diagonals:\n", np.round(np.array([np.diag(S) for S in covs_b8]), 3))

▶ What you'll see: class 0 spreads mainly in x1, while class 1 spreads mainly in x2.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 2.8))  # two covariance heatmaps.
for k_b8 in range(2):
    ax[k_b8].imshow(covs_b8[k_b8], cmap="viridis"); ax[k_b8].set_title(f"class {k_b8} covariance")
plt.suptitle("Basic 8: QDA keeps separate shapes"); plt.show()

▶ What you'll see: the two covariance heatmaps differ, which is exactly what QDA preserves.

👀 Takeaway: QDA spends more parameters so each class can have its own geometry.

### Basic 9 — Score one point with QDA

**Goal.** Use class-specific covariance in the log Gaussian score, because QDA classification depends on shape as well as center. We build it in 3 steps.

In [ ]:
X_b9 = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2], [-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # data.
y_b9 = np.array([0] * 6 + [1] * 6)  # labels.
_, mus_b9, covs_b9, priors_b9 = fit_gaussian_params(X_b9, y_b9)  # QDA parameters.
print("means:\n", np.round(mus_b9, 3))

▶ What you'll see: the two class centers are separated vertically.

In [ ]:
x_b9 = np.array([0.75, 1.1])  # point where covariance shape matters.
scores_b9 = qda_scores(x_b9, mus_b9, covs_b9, priors_b9)[0]  # QDA scores.
print("QDA scores:", np.round(scores_b9, 3), "prediction:", int(np.argmax(scores_b9)))
assert scores_b9.shape == (2,)

▶ What you'll see: QDA returns one log score per class, and the larger one is the prediction.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["class 0", "class 1"], scores_b9, color=["steelblue", "darkorange"])
plt.title("Basic 9: QDA score comparison"); plt.ylabel("log score"); plt.show()

▶ What you'll see: shape-aware scoring can favor the class whose covariance makes the point less surprising.

👀 Takeaway: QDA is Gaussian log scoring with a different covariance matrix in each class.

### Basic 10 — Convert scores to a prediction

**Goal.** Turn discriminant scores into a label and confidence-like softmax probabilities, because the largest log score determines the class. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([-3.2, -1.7])  # two log scores for one point.
pred_b10 = int(np.argmax(scores_b10))  # winner by largest log score.
shifted_b10 = scores_b10 - scores_b10.max()  # stabilize exponentials.
probs_b10 = np.exp(shifted_b10) / np.exp(shifted_b10).sum()  # softmax probabilities.
print("prediction:", pred_b10, "probabilities:", np.round(probs_b10, 3))
assert pred_b10 == 1 and np.round(probs_b10[1], 3) == 0.818

▶ What you'll see: class 1 wins, and its normalized probability is about 0.818.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["class 0", "class 1"], probs_b10, color=["steelblue", "darkorange"])
plt.title("Basic 10: normalized score weights"); plt.ylabel("softmax of log scores"); plt.ylim(0, 1); plt.show()

▶ What you'll see: exponentiating shifted log scores gives readable class weights without changing the winner.

👀 Takeaway: predictions use `argmax`, while a shifted softmax is a convenient way to inspect relative score strength.

## 🟡 Easy

### Easy 1 — Fit LDA end to end

**Goal.** Estimate parameters, score training points, and measure training error, because LDA is a full classifier once the shared covariance is known. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # easy separated data.
y_e1 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
_, mus_e1, covs_e1, priors_e1 = fit_gaussian_params(X_e1, y_e1)  # means and priors.
Sp_e1 = pooled_covariance(X_e1, y_e1)  # shared covariance.
print("mus:\n", np.round(mus_e1, 3))

▶ What you'll see: two fitted centers, one for each class.

In [ ]:
scores_e1 = lda_scores(X_e1, mus_e1, Sp_e1, priors_e1)  # score every training point.
preds_e1 = np.argmax(scores_e1, axis=1)  # predicted labels.
err_e1 = np.mean(preds_e1 != y_e1)  # training error.
print("predictions:", preds_e1, "training error:", err_e1)
assert err_e1 == 0.0

▶ What you'll see: all training labels are recovered on this clean toy data.

In [ ]:
draw_boundary(X_e1, y_e1, lambda G: lda_scores(G, mus_e1, Sp_e1, priors_e1), "Easy 1: LDA end-to-end boundary")

▶ What you'll see: a straight LDA boundary separating the two clusters.

👀 Takeaway: fitting LDA means estimating class means, one pooled covariance, priors, then taking the largest linear score.

### Easy 2 — Fit QDA end to end

**Goal.** Fit QDA on classes with different shapes, because separate covariances can capture geometry LDA deliberately ignores. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2], [-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # two different shapes.
y_e2 = np.array([0] * 6 + [1] * 6)  # labels.
_, mus_e2, covs_e2, priors_e2 = fit_gaussian_params(X_e2, y_e2)  # QDA summaries.
print("cov diag:\n", np.round(np.array([np.diag(S) for S in covs_e2]), 3))

▶ What you'll see: the covariance diagonals reveal different spread directions.

In [ ]:
scores_e2 = qda_scores(X_e2, mus_e2, covs_e2, priors_e2)  # QDA training scores.
preds_e2 = np.argmax(scores_e2, axis=1)  # predicted labels.
err_e2 = np.mean(preds_e2 != y_e2)  # training error.
print("QDA training error:", err_e2)
assert err_e2 == 0.0

▶ What you'll see: QDA fits this clean shape-different data perfectly.

In [ ]:
draw_boundary(X_e2, y_e2, lambda G: qda_scores(G, mus_e2, covs_e2, priors_e2), "Easy 2: QDA curved boundary")

▶ What you'll see: the boundary bends to account for the different covariance shapes.

👀 Takeaway: QDA is more flexible because class-specific covariance keeps shape information.

### Easy 3 — Compare LDA and QDA boundaries

**Goal.** Put LDA and QDA on the same shape-different dataset, because the shared-covariance assumption is visible in the boundary geometry. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2], [-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # same data.
y_e3 = np.array([0] * 6 + [1] * 6)  # labels.
_, mus_e3, covs_e3, priors_e3 = fit_gaussian_params(X_e3, y_e3)  # Gaussian parameters.
Sp_e3 = pooled_covariance(X_e3, y_e3)  # LDA shared covariance.
print("pooled diag:", np.round(np.diag(Sp_e3), 3))

▶ What you'll see: the pooled covariance averages the two distinct shapes.

In [ ]:
probe_e3 = np.array([[0.6, 0.8], [0.0, 0.9], [0.8, 1.6]])  # ambiguous test points.
lda_pred_e3 = np.argmax(lda_scores(probe_e3, mus_e3, Sp_e3, priors_e3), axis=1)  # LDA predictions.
qda_pred_e3 = np.argmax(qda_scores(probe_e3, mus_e3, covs_e3, priors_e3), axis=1)  # QDA predictions.
print("LDA preds:", lda_pred_e3, "QDA preds:", qda_pred_e3)
assert len(lda_pred_e3) == 3 and len(qda_pred_e3) == 3

▶ What you'll see: the two models may disagree near the boundary because they encode different shape assumptions.

In [ ]:
draw_boundary(X_e3, y_e3, lambda G: lda_scores(G, mus_e3, Sp_e3, priors_e3), "Easy 3a: LDA straight boundary")
draw_boundary(X_e3, y_e3, lambda G: qda_scores(G, mus_e3, covs_e3, priors_e3), "Easy 3b: QDA curved boundary")

▶ What you'll see: LDA gives one straight line; QDA gives a curved boundary on the same points.

👀 Takeaway: LDA versus QDA is mainly a bias-variance choice about whether covariance shapes should be shared.

### Easy 4 — Add the lesson's selection cost

**Goal.** Recompute the verified raw score, cost, gap, and stabilized score, because model selection should compare complete decision scores. We build it in 3 steps.

In [ ]:
losses_e4 = np.array([0.268, 0.148, 0.522])  # verified per-example losses.
raw_e4 = round(float(losses_e4.mean()), 3)  # empirical risk rounded to the lesson's displayed scale.
cost_e4 = 0.100  # complexity or operational cost.
score_e4 = raw_e4 + cost_e4  # full baseline score.
print("raw:", round(raw_e4, 3), "score:", round(score_e4, 3))
assert round(raw_e4, 3) == 0.313 and round(score_e4, 3) == 0.413

▶ What you'll see: the raw average is not the full model-selection score.

In [ ]:
flex_e4 = 0.457  # alternative score.
stable_e4 = 0.80 * score_e4  # stabilizing knob.
gap_e4 = flex_e4 - score_e4  # evidence gap.
print("gap:", round(gap_e4, 3), "stable:", round(stable_e4, 3))
assert round(gap_e4, 3) == 0.044 and round(stable_e4, 3) == 0.330

▶ What you'll see: the stabilized score is lower than both baseline and flexible alternative.

In [ ]:
vals_e4 = np.array([score_e4, flex_e4, stable_e4])  # comparable candidates.
plt.figure(figsize=(4, 3)); plt.bar(["baseline", "flexible", "stable"], vals_e4, color=["gray", "darkorange", "seagreen"])
plt.title("Easy 4: full score comparison"); plt.ylabel("lower is better"); plt.show()
print("best index:", int(np.argmin(vals_e4)))

▶ What you'll see: the lowest bar is the stabilized option.

👀 Takeaway: selection uses the score scale defined by the method, including costs and stability adjustments.

### Easy 5 — Use priors to reflect class imbalance

**Goal.** Compare equal priors with imbalanced priors, because priors should shift predictions when class prevalence is real. We build it in 3 steps.

In [ ]:
X_e5 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # balanced geometry.
y_e5 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
_, mus_e5, covs_e5, priors_e5 = fit_gaussian_params(X_e5, y_e5)  # summaries.
Sp_e5 = pooled_covariance(X_e5, y_e5)  # shared covariance.
print("empirical priors:", priors_e5)

▶ What you'll see: equal counts produce equal empirical priors.

In [ ]:
x_e5 = np.array([2.0, 1.9])  # near-boundary point.
equal_e5 = lda_scores(x_e5, mus_e5, Sp_e5, np.array([0.5, 0.5]))[0]  # equal priors.
skew_e5 = lda_scores(x_e5, mus_e5, Sp_e5, np.array([0.8, 0.2]))[0]  # skewed priors.
print("equal margin:", round(equal_e5[0] - equal_e5[1], 3), "skew margin:", round(skew_e5[0] - skew_e5[1], 3))
assert round((skew_e5[0] - skew_e5[1]) - (equal_e5[0] - equal_e5[1]), 3) == round(np.log(0.8 / 0.2), 3)

▶ What you'll see: the class-0-vs-class-1 margin increases by the log prior odds change.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["equal priors", "skewed priors"], [equal_e5[0] - equal_e5[1], skew_e5[0] - skew_e5[1]], color=["gray", "seagreen"])
plt.axhline(0, color="black", linewidth=1); plt.title("Easy 5: prior-driven margin shift"); plt.ylabel("class0 - class1 score"); plt.show()

▶ What you'll see: the skewed-prior bar is higher, meaning class 0 gets a prevalence advantage.

👀 Takeaway: priors are part of the model, so they must match the population where predictions will be used.

## 🔴 Advanced

### Advanced 1 — Regularize a nearly singular covariance

**Goal.** Stabilize a covariance inverse by adding a small diagonal value, because QDA can fail when a class has too few or too-collinear examples. We build it in 3 steps.

In [ ]:
X0_a1 = np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 2.0]])  # collinear points make covariance singular.
S_a1 = np.cov(X0_a1, rowvar=False, bias=False)  # rank-deficient covariance.
det_a1 = np.linalg.det(S_a1)  # determinant reveals singularity.
print("covariance:\n", S_a1, "det:", round(det_a1, 6))
assert round(det_a1, 6) == 0.0

▶ What you'll see: the determinant is zero, so a plain inverse is not available.

In [ ]:
reg_a1 = 0.1  # diagonal loading amount.
Sreg_a1 = S_a1 + reg_a1 * np.eye(2)  # stabilized covariance.
cond_a1 = np.linalg.cond(Sreg_a1)  # condition number after regularization.
print("regularized det:", round(np.linalg.det(Sreg_a1), 3), "condition:", round(cond_a1, 3))
assert np.linalg.det(Sreg_a1) > 0

▶ What you'll see: diagonal loading makes the covariance invertible.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["det before", "det after"], [det_a1, np.linalg.det(Sreg_a1)], color=["red", "seagreen"])
plt.title("Advanced 1: covariance regularization"); plt.ylabel("determinant"); plt.show()

▶ What you'll see: the determinant rises from zero to a positive value.

👀 Takeaway: regularizing covariance is the QDA version of the lesson's stability knob.

### Advanced 2 — Sweep QDA regularization on validation data

**Goal.** Choose the covariance regularization strength by validation error, because the best-looking training model may not be the most durable one. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[-1.1, -0.1], [-0.7, 0.2], [-0.2, -0.2], [0.2, 0.1], [0.7, -0.1], [1.1, 0.2], [-0.2, 1.0], [0.1, 1.3], [0.0, 1.7], [0.2, 2.0], [-0.1, 2.3], [0.1, 2.6]])  # shape-different data.
y_a2 = np.array([0] * 6 + [1] * 6)  # labels.
val_idx_a2 = np.array([1, 8])  # one held-out point per class.
train_mask_a2 = np.ones(len(y_a2), dtype=bool); train_mask_a2[val_idx_a2] = False  # training mask.
print("validation labels:", y_a2[val_idx_a2])

▶ What you'll see: validation contains one class-0 and one class-1 example.

In [ ]:
regs_a2 = np.array([0.0, 0.02, 0.1, 0.4])  # regularization grid.
val_errs_a2 = []  # validation error per setting.
train_errs_a2 = []  # training error per setting.
_, mus_a2, covs_a2, priors_a2 = fit_gaussian_params(X_a2[train_mask_a2], y_a2[train_mask_a2])  # fit once from training data.
print("regs:", regs_a2)

▶ What you'll see: four diagonal-loading strengths will be compared.

In [ ]:
for reg_a2 in regs_a2:  # evaluate each regularization value.
    pred_train_a2 = np.argmax(qda_scores(X_a2[train_mask_a2], mus_a2, covs_a2, priors_a2, reg=reg_a2), axis=1)  # train predictions.
    pred_val_a2 = np.argmax(qda_scores(X_a2[val_idx_a2], mus_a2, covs_a2, priors_a2, reg=reg_a2), axis=1)  # validation predictions.
    train_errs_a2.append(np.mean(pred_train_a2 != y_a2[train_mask_a2]))  # train error.
    val_errs_a2.append(np.mean(pred_val_a2 != y_a2[val_idx_a2]))  # validation error.
print("train errors:", train_errs_a2, "validation errors:", val_errs_a2)
assert len(val_errs_a2) == 4

▶ What you'll see: every regularization value has a train and validation score on the same scale.

In [ ]:
best_reg_a2 = regs_a2[int(np.argmin(val_errs_a2))]  # choose lowest validation error.
plt.figure(figsize=(5, 3)); plt.plot(regs_a2, train_errs_a2, marker="o", label="train")
plt.plot(regs_a2, val_errs_a2, marker="s", label="validation"); plt.axvline(best_reg_a2, color="red", linestyle="--", label="best")
plt.title("Advanced 2: validation chooses QDA regularization"); plt.xlabel("diagonal loading"); plt.ylabel("error rate"); plt.legend(); plt.show()
print("best reg:", best_reg_a2)

▶ What you'll see: the selected regularization is the first value with the lowest validation error.

👀 Takeaway: covariance flexibility should be tuned with held-out data, not just trusted because it fits training points.

### Advanced 3 — Count LDA versus QDA parameters

**Goal.** Compare model capacity numerically, because QDA's curved boundary comes from estimating many more covariance parameters. We build it in 2 steps.

In [ ]:
p_a3 = 5  # number of features.
K_a3 = 3  # number of classes.
mean_params_a3 = K_a3 * p_a3  # one mean vector per class.
prior_params_a3 = K_a3 - 1  # priors sum to one.
cov_params_one_a3 = p_a3 * (p_a3 + 1) // 2  # symmetric covariance entries.
lda_params_a3 = mean_params_a3 + prior_params_a3 + cov_params_one_a3  # shared covariance.
qda_params_a3 = mean_params_a3 + prior_params_a3 + K_a3 * cov_params_one_a3  # one covariance per class.
print("LDA params:", lda_params_a3, "QDA params:", qda_params_a3)
assert lda_params_a3 == 32 and qda_params_a3 == 62

▶ What you'll see: QDA uses almost twice as many parameters in this 5-feature, 3-class example.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["LDA", "QDA"], [lda_params_a3, qda_params_a3], color=["steelblue", "darkorange"])
plt.title("Advanced 3: parameter count"); plt.ylabel("estimated parameters"); plt.show()

▶ What you'll see: QDA's bar is taller because every class gets its own covariance matrix.

👀 Takeaway: QDA's extra flexibility is a variance cost that needs enough data or regularization.

### Advanced 4 — Detect when priors drift

**Goal.** Compare training priors with deployment priors, because a classifier calibrated on one population can be biased in another. We build it in 3 steps.

In [ ]:
train_counts_a4 = np.array([80, 20])  # training population is class-0 heavy.
deploy_counts_a4 = np.array([50, 50])  # future population is balanced.
train_prior_a4 = train_counts_a4 / train_counts_a4.sum()  # training priors.
deploy_prior_a4 = deploy_counts_a4 / deploy_counts_a4.sum()  # deployment priors.
print("train priors:", train_prior_a4, "deploy priors:", deploy_prior_a4)
assert np.allclose(train_prior_a4, [0.8, 0.2]) and np.allclose(deploy_prior_a4, [0.5, 0.5])

▶ What you'll see: the class prevalence changed substantially between train and deployment.

In [ ]:
log_odds_shift_a4 = np.log(deploy_prior_a4[0] / deploy_prior_a4[1]) - np.log(train_prior_a4[0] / train_prior_a4[1])  # prior log-odds correction.
print("log-odds correction for class0-vs-class1:", round(log_odds_shift_a4, 3))
assert round(log_odds_shift_a4, 3) == -1.386

▶ What you'll see: class 0 should lose 1.386 log-odds units when moving to balanced deployment.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["train log odds", "deploy log odds"], [np.log(0.8/0.2), np.log(0.5/0.5)], color=["red", "seagreen"])
plt.title("Advanced 4: prior drift changes margins"); plt.ylabel("log prior odds class0/class1"); plt.show()

▶ What you'll see: the prior advantage for class 0 disappears under balanced deployment.

👀 Takeaway: priors encode population assumptions, so prior drift is a real model-monitoring issue.

### Advanced 5 — Visualize LDA as a one-dimensional projection

**Goal.** Project points onto the LDA direction, because the two-class LDA boundary can be read as a threshold on one linear score. We build it in 3 steps.

In [ ]:
X_a5 = np.array([[1.0, 1.2], [1.4, 0.8], [1.2, 1.0], [0.8, 1.1], [3.0, 2.8], [3.4, 3.2], [2.9, 3.5], [3.2, 3.0]])  # separated data.
y_a5 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
_, mus_a5, covs_a5, priors_a5 = fit_gaussian_params(X_a5, y_a5)  # class summaries.
Sp_a5 = pooled_covariance(X_a5, y_a5)  # shared covariance.
w_a5 = np.linalg.inv(Sp_a5) @ (mus_a5[1] - mus_a5[0])  # LDA separating direction.
print("LDA direction:", np.round(w_a5, 3))

▶ What you'll see: a two-number vector defining the direction on which LDA separates classes.

In [ ]:
proj_a5 = X_a5 @ w_a5  # one-dimensional coordinates.
center0_a5 = mus_a5[0] @ w_a5  # projected class-0 mean.
center1_a5 = mus_a5[1] @ w_a5  # projected class-1 mean.
threshold_a5 = 0.5 * (center0_a5 + center1_a5)  # equal-prior midpoint in projection space for this display.
print("projected centers:", round(center0_a5, 3), round(center1_a5, 3), "threshold:", round(threshold_a5, 3))
assert center1_a5 > center0_a5

▶ What you'll see: the class centers are well separated after projection.

In [ ]:
plt.figure(figsize=(5, 2.8)); plt.scatter(proj_a5[y_a5 == 0], np.zeros(np.sum(y_a5 == 0)), color="steelblue", label="class 0")
plt.scatter(proj_a5[y_a5 == 1], np.zeros(np.sum(y_a5 == 1)), color="darkorange", label="class 1")
plt.axvline(threshold_a5, color="black", linestyle="--", label="threshold")
plt.yticks([]); plt.title("Advanced 5: LDA projection view"); plt.xlabel("x · w"); plt.legend(); plt.show()

▶ What you'll see: the two classes separate on a single line, with a threshold between their projected centers.

👀 Takeaway: two-class LDA is a Gaussian-derived linear classifier, and its decision can be understood as projecting onto one discriminant direction.